In [1]:
import pandas as pd

In [2]:
dfs = pd.read_excel('../Data/raw/2007-2024-PIT-Counts-by-CoC.xlsb', sheet_name=None)
for name, sheets in dfs.items():
    print(name)


2024
2023
2022
2021
2020
2019
2018
2017
2016
2015
2014
2013
2012
2011
2010
2009
2008
2007
CoC Mergers
CoCs, States, DDs
field list, all years
Table_categories
Generate_Table
PIT_Counts_Table_Template
PIT_Counts_Table
Generate_Chart
Chart_categories
PIT_Counts_Chart
PIT_Counts_Chart_Template


In [3]:
dfs2 = list()
for name, sheets in dfs.items():
    try:
        year = int(name)
        if  year >= 2009 and year <= 2024:
            df = sheets
            df["year"] = year
            dfs2.append(df)
    except ValueError:
        print("Not a year:", name)

Not a year: CoC Mergers
Not a year: CoCs, States, DDs
Not a year: field list, all years
Not a year: Table_categories
Not a year: Generate_Table
Not a year: PIT_Counts_Table_Template
Not a year: PIT_Counts_Table
Not a year: Generate_Chart
Not a year: Chart_categories
Not a year: PIT_Counts_Chart
Not a year: PIT_Counts_Chart_Template


In [4]:
for df in dfs2:
    print(df['year'].iat[0])
    print(df.shape)
df = dfs2[1]
col_list = df.columns
sorted_list = sorted(col_list, key=len)
for val in sorted_list:
    print(val)
#for val in my_list:
    #print(val)
#search_strings = [ "Code", "Name", "Total Homeless", "Total Sheltered", "Total Unsheltered"]
#indices = [i for i, s in enumerate(my_list) if "Code" in s]




2024
(390, 1309)
2023
(388, 656)
2022
(388, 577)
2021
(388, 543)
2020
(387, 543)
2019
(386, 543)
2018
(387, 543)
2017
(387, 543)
2016
(385, 506)
2015
(386, 386)
2014
(383, 348)
2013
(383, 95)
2012
(382, 48)
2011
(383, 48)
2010
(384, 32)
2009
(384, 28)
year
CoC Name
CoC Number
Count Types
Overall Homeless
Unsheltered Homeless
Sheltered ES Homeless
Sheltered TH Homeless
Sheltered SH Homeless
Overall Homeless - Man
Overall Homeless - Woman
Overall Homeless - White
Sheltered Total Homeless
Overall Homeless Veterans
Overall Homeless - Over 64
Unsheltered Homeless - Man
Overall Homeless - Under 18
Sheltered ES Homeless - Man
Sheltered TH Homeless - Man
Sheltered SH Homeless - Man
Unsheltered Homeless - Woman
Unsheltered Homeless - White
Overall Homeless Individuals
Overall Chronically Homeless
Overall Homeless - Non Binary
Sheltered ES Homeless - Woman
Sheltered ES Homeless - White
Sheltered TH Homeless - Woman
Sheltered TH Homeless - White
Sheltered SH Homeless - Woman
Sheltered SH Homeless

## Possible Important Columns
* CoC_Code
* CoC_Name
* Total_Homeless_Persons
* Total_Sheltered_Persons
* Total_Unsheltered_Persons
* Adults
* Children
* Youth
* Families_with_Children
* Individual_Persons
* Chronically_Homeless
* Veterans
* Unaccompanied_Youth
* Race/Ethnicity
* Gender
* Disabilities
* Domestic_Violence_Victims
* Traumatic_Brain_Injury
* Substance_Abuse
* HIV/AIDS

## Mapped Columns
* CoC Name
* CoC Number
* Count Types
* Overall Homeless
* Unsheltered Homeless
* Sheltered ES Homeless
* Sheltered TH Homeless
* Sheltered SH Homeless
* Sheltered Total Homeless
* Overall Homeless Individuals
* Overall Homeless People in Families

In [5]:
coc_columns = [
    "CoC Name",
    "CoC Number",
    "year",
    "Count Types",
    "Overall Homeless",
    "Unsheltered Homeless",
 #   "Sheltered ES Homeless",
 #   "Sheltered TH Homeless",
 #   "Sheltered SH Homeless",
    "Sheltered Total Homeless",
    "Overall Homeless Individuals",
    "Overall Homeless People in Families"
]

dfs3 = list()
for df in dfs2:
    try:
        # Pass the list of column names directly to the DataFrame
        slice_df = df[coc_columns].copy()
        print(len(slice_df.columns))
        dfs3.append(slice_df)
    except KeyError as e:
            # The exception object 'e' now holds the key
            missing_key = e.args[0]
            print(f"Skipping a DataFrame from year {df['year'].iat[0]} because of a missing column: '{missing_key}'")

coc_dataset = pd.concat(dfs3)
coc_dataset.head()
coc_dataset.to_csv('coc_dataset.csv', index=False)

9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9


In [6]:
coc_dataset.isna().sum()
len(coc_dataset)

6171

## Drop Erroneous Rows

Some rows in the excel sheet had file and text information. Not CoC counts. I delete these rows here.

In [7]:
# Calculate the number of NA's in each row
na_counts = coc_dataset.isnull().sum(axis=1)
# Create a boolean Series: True for rows to keep (4 or fewer NA's)
rows_to_keep = na_counts <= 4

# Create the cleaned DataFrame
df_cleaned = coc_dataset[rows_to_keep]

# Create the dropped DataFrame by inverting the boolean Series
df_dropped = coc_dataset[~rows_to_keep]
df_dropped

coc_dataset = df_cleaned.copy()


## Duplicate CoCs
checking if any CoCs are duplicates in the year

In [8]:
for year in range(2009, 2025):
    sliced_year = coc_dataset[ coc_dataset["year"] == year ]
    if not sliced_year.empty:
        unique_cocs = sliced_year['CoC Name'].nunique()
        print(f"{year}:\t{unique_cocs}\tno dups:\t{unique_cocs == len(sliced_year)}")

2009:	382	no dups:	True
2010:	382	no dups:	True
2011:	381	no dups:	True
2012:	380	no dups:	True
2013:	381	no dups:	True
2014:	381	no dups:	True
2015:	384	no dups:	True
2016:	383	no dups:	True
2017:	385	no dups:	True
2018:	385	no dups:	True
2019:	384	no dups:	True
2020:	385	no dups:	True
2021:	386	no dups:	True
2022:	386	no dups:	True
2023:	385	no dups:	True
2024:	386	no dups:	True


## Abnormal CoCs
Getting a list of CoCs that are not in every year. Therefore they were added or subtracted during this time period.

In [10]:
df = coc_dataset[ coc_dataset["year"] >= 2009 ]

# how many times a CoC is repeated if it is in all
# years. i.e. the number of years.
years = df["year"].unique()

# count how many times a year is applied to each CoC
# ie the number of times that CoC is repeated in the dataset
coc_counts = df.groupby("CoC Name")['year'].nunique()

# if the times a CoC is repeated is less than
# every year then that coc isn't in every year
# get a list of the indexes of the groupby 'CoC Name'
# i.e. get each CoC Name that isn't in every year
abnormal_cocs = coc_counts[coc_counts < len(years)].index

# get a dataframe with all the abnormal CoCs in it
abnormal = df[ df["CoC Name"].isin(abnormal_cocs) ]
print(len(abnormal))

# get a dataframe with all normal CoCs in it
normal = df[ ~df["CoC Name"].isin(abnormal_cocs) ]

# save stable cocs to a file
normal.to_csv("stable_cocs_2009_2024.csv", index=False)

# print out abnormal cocs
abnormal.groupby("CoC Number")['year'].apply(list)

184


CoC Number
AL-508                                               [2024]
AR-502                                               [2009]
AR-503    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
AR-504                                         [2010, 2009]
AR-506                                               [2009]
AR-508    [2024, 2023, 2022, 2021, 2019, 2018, 2017, 201...
AR-512                                               [2010]
CA-527    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
CA-528                                   [2012, 2011, 2010]
CA-529    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
CA-530    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
CA-531                 [2024, 2023, 2022, 2021, 2020, 2019]
CO-505                       [2024, 2023, 2022, 2021, 2020]
GA-502    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
GA-508    [2024, 2023, 2022, 2021, 2020, 2019, 2018, 201...
MA-501                                   [2011, 2010, 2009]
MP-500                 [2022,